<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="https://static.vecteezy.com/system/resources/thumbnails/021/747/103/small_2x/banner-for-mental-health-concept-illustration-design-of-human-brain-made-of-flowers-generative-ai-photo.jpg" alt="Mental Health" style="width: 600px">
</div>

### Preparacion Ambiente

Notebook de preparacion de ambiente

In [0]:
dbutils.widgets.removeAll()
dbutils.widgets.text("storageName","saafadproject")
storageName = dbutils.widgets.get("storageName")

In [0]:
spark.sql(f"""CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-raw`
URL 'abfss://raw@{storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL credential)
COMMENT 'Ubicación externa para las tablas raw del Data Lake';""")

spark.sql(f"""CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-bronze`
URL 'abfss://bronze@{storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL credential)
COMMENT 'Ubicación externa para las tablas bronze del Data Lake';""")

spark.sql(f"""CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-silver`
URL 'abfss://silver@{storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL credential)
COMMENT 'Ubicación externa para las tablas silver del Data Lake';""")

spark.sql(f"""CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-gold`
URL 'abfss://gold@{storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL credential)
COMMENT 'Ubicación externa para las tablas gold del Data Lake';""")

In [0]:
catalog_name = "health_db_catalog"
spark.sql(f"DROP CATALOG IF EXISTS {catalog_name} CASCADE;")
spark.sql(f"""CREATE CATALOG IF NOT EXISTS {catalog_name}
COMMENT 'Catalogo para la arquitectura medallion del ambiente en dev';""")

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS health_db_catalog.raw;")
spark.sql("CREATE SCHEMA IF NOT EXISTS health_db_catalog.bronze;")
spark.sql("CREATE SCHEMA IF NOT EXISTS health_db_catalog.silver;")
spark.sql("CREATE SCHEMA IF NOT EXISTS health_db_catalog.gold;")

In [0]:
spark.sql(f"""
          CREATE OR REPLACE TABLE health_db_catalog.bronze.us_state_codes
          (
            id INTEGER,
            state STRING
          )
          USING DELTA
          LOCATION 'abfss://bronze@{storageName}.dfs.core.windows.net/us_state_codes'
          """)

In [0]:
spark.sql(f"""
          CREATE OR REPLACE TABLE health_db_catalog.bronze.patients_information
          (
            person_id INTEGER,
            gender STRING,
            age INTEGER,
            occupation STRING,
            state INTEGER
          )
          USING DELTA
          LOCATION 'abfss://bronze@{storageName}.dfs.core.windows.net/patients_information'
          """)

In [0]:
spark.sql(f"""
          CREATE OR REPLACE TABLE health_db_catalog.bronze.daily_nutrition_facts
          (
            person_id INTEGER,
            dialy_vitamins STRING
          )
          USING DELTA
          LOCATION 'abfss://bronze@{storageName}.dfs.core.windows.net/daily_nutrition_facts'
          """)

In [0]:
spark.sql(f"""
          CREATE OR REPLACE TABLE health_db_catalog.bronze.temperature
          (
            state_id INTEGER,
            state STRING,
            month INTEGER,
            year INTEGER,
            avg_temp DECIMAL(3,1)
          )
          USING DELTA
          LOCATION 'abfss://bronze@{storageName}.dfs.core.windows.net/temperature'
          """)

In [0]:
spark.sql(f"""
          CREATE OR REPLACE TABLE health_db_catalog.bronze.healthy_lifestyle
          (
            person_id INTEGER,
            sleep_duration DECIMAL(2,1),
            quality_of_sleep INTEGER,
            physical_activity_level INTEGER,
            stress_level INTEGER,
            BMI_category STRING,
            blood_pressure STRING,
            hearth_rate INTEGER,
            daily_steps INTEGER,
            sleep_disorder STRING
          )
          USING DELTA
          LOCATION 'abfss://bronze@{storageName}.dfs.core.windows.net/healthy_lifestyle'
          """)